# 🔒 Módulo 04 — Aprobación de Herramientas (Human-in-the-Loop)

> **Objetivo:** Aprender cómo hacer que ciertas acciones requieran confirmación humana antes de ejecutarse.

## ¿Por qué Human-in-the-Loop?

Hay acciones que **no quieres que un agente ejecute autónomamente**:
- Enviar emails
- Borrar registros
- Hacer transacciones financieras
- Modificar configuración de producción

El patrón HITL pausa la ejecución y pide aprobación al usuario antes de continuar.

## Flujo

```
Usuario: "Manda un email a X"
   ↓
Agente: quiero invocar send_email(...)
   ↓
Framework: ⚠️ ese tool requiere aprobación → result.user_input_requests
   ↓
Tú (código de UI): muestras la solicitud al usuario y obtienes y/n
   ↓
Reenvías al agente la respuesta de aprobación
   ↓
Agente ejecuta send_email y continúa
```

## API

```python
@tool(approval_mode="always_require")
def send_email(...): ...

result = await agent.run(query)
while result.user_input_requests:
    # construir inputs con la respuesta de aprobación
    result = await agent.run(new_inputs)
```


## ⚙️ Setup inicial

Esta celda carga la configuración de Azure OpenAI desde `appsettings.Development.json`
(o variables de entorno) y crea el cliente que reutilizaremos durante todo el notebook.

> 💡 Solo necesitas ejecutar esta celda **una vez** al abrir el notebook.


In [ ]:
# Aseguramos que la raíz del proyecto esté en sys.path para importar `helpers`
import sys
from pathlib import Path

_ROOT = Path.cwd()
# Si abriste el notebook desde la carpeta notebooks/, sube un nivel
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from agent_framework import Agent
from helpers.config import create_chat_client
print("✅ Helpers cargados. Endpoint:", create_chat_client().__class__.__name__)


## 1️⃣ Definir tools sensibles y un helper de aprobación

Marca con `approval_mode="always_require"` las funciones que necesitan confirmación.


In [ ]:
from typing import Annotated, Any
from agent_framework import Message, tool
from pydantic import Field


@tool(approval_mode="always_require")
def send_email(
    to: Annotated[str, Field(description="Recipient email address")],
    subject: Annotated[str, Field(description="Email subject")],
    body: Annotated[str, Field(description="Email body content")],
) -> str:
    """Sends an email to a specified recipient."""
    return f"Email sent successfully to {to} with subject '{subject}'"


@tool(approval_mode="always_require")
def delete_record(
    record_id: Annotated[int, Field(description="The record ID to delete")],
) -> str:
    """Deletes a record from the database by its ID."""
    return f"Record {record_id} has been permanently deleted"


@tool(approval_mode="never_require")
def search_database(
    query: Annotated[str, Field(description="Search query")],
) -> str:
    """Searches the database for records matching a query."""
    return f"Found 5 results for: {query}"


async def auto_approve_all(agent, query: str, approve: bool = True):
    """Helper que aprueba (o rechaza) automáticamente todas las solicitudes."""
    result = await agent.run(query)

    while getattr(result, "user_input_requests", None):
        new_inputs: list[Any] = [query]
        for req in result.user_input_requests:
            print(
                f"   🔔 Solicitud de aprobación → función: {req.function_call.name}"
                f"  · args: {req.function_call.arguments}"
            )
            new_inputs.append(Message("assistant", [req]))
            new_inputs.append(
                Message("user", [req.to_function_approval_response(approve)])
            )
        result = await agent.run(new_inputs)

    return result


print("✅ Tools y helper definidos. En producción reemplaza `auto_approve_all` por una UI real.")


## 2️⃣ Caso 1 — Envío de email requiere confirmación

El agente solicita aprobación, el helper auto-aprueba (en producción sería un click del usuario).


In [ ]:
agent = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un asistente de correo electrónico. Cuando te pidan enviar un correo, "
        "usa la herramienta send_email."
    ),
    tools=[send_email],
)

query = (
    "Send an email to workshop@example.com with subject 'Workshop Reminder' "
    "and body 'Don't forget the Agent Framework workshop tomorrow!'"
)

result = await auto_approve_all(agent, query, approve=True)
print(f"\n✅ Respuesta final: {result.text}")


## 3️⃣ Caso 2 — Mezcla de tools normales y con aprobación

Sólo las acciones críticas (`delete_record`) piden confirmación. La búsqueda (`search_database`) se ejecuta directamente.


In [ ]:
agent = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un asistente de base de datos. Usa search_database para consultas "
        "y delete_record para eliminaciones. Siempre busca antes de eliminar."
    ),
    tools=[search_database, delete_record],
)

# Búsqueda — NO requiere aprobación
search = await agent.run("Search for users named John")
print(f"✅ Búsqueda (sin aprobación): {search.text}\n")

# Eliminación — REQUIERE aprobación
print("🔒 Solicitando eliminación (requiere aprobación)...")
delete = await auto_approve_all(agent, "Delete record 42", approve=True)
print(f"\n✅ Eliminación (con aprobación): {delete.text}")


## 🎯 Resumen

| Capacidad | Cómo se hace |
|-----------|--------------|
| Marcar tool como "requiere aprobación" | `@tool(approval_mode="always_require")` |
| Detectar solicitudes | `result.user_input_requests` (lista) |
| Aprobar/rechazar | `Message("user", [req.to_function_approval_response(True)])` |
| Continuar | Reejecutar `agent.run(new_inputs)` con la respuesta de aprobación |

> 💡 En producción, cuando recibes `user_input_requests`, presenta la información al usuario
> en tu UI (Teams card, web modal, etc.) y persiste el estado hasta recibir la decisión.

**Siguiente módulo →** [Módulo 05 — Multimodal (Análisis de Imágenes)](./05_multimodal.ipynb)
